In [ ]:
import os, sys, subprocess, pathlib
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats
from matplotlib.lines import Line2D
from IPython.display import display

assert pathlib.Path.cwd().name == 'Comparisons', 'run this notebook from the Comparisons/ folder'
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
from calculations.figures import results_dir, save_figure
from calculations.permeability import UNITS_SIEVERTS, UNITS_HENRY
FIGS = results_dir("study-comparison")                  # sweep CSVs, figures and fit tables go here

# The studies to compare. Every value used for a study comes from its own study_config.py.
STUDIES = ['Guo_etal_2025', 'Fuerst_et_al_2024', 'incoloy802_cr2o3']

SHORT_NAMES = {   # metal key in a study_config.py -> label on the figures; any other metal shows its key
    '316L_Guo_2025':                                   '316L',
    'Hastelloy_N_fuerst_2024':                         'Hastelloy N',
    'metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt1985': 'Incoloy 802',
}
REFRESH = False   # True recomputes every study. Otherwise a study is recomputed only when its CSV is
                  # missing or older than its study_config.py, compare_worker.py or the model code.
R = 8.314         # J/(mol K)

In [ ]:
WORKER = pathlib.Path('compare_worker.py').resolve()
sweep_csv = lambda study: FIGS / f'{study}_sweeps.csv'
CODE = [WORKER, *(ROOT / 'calculations').rglob('*.py')]

def stale(study):
    """Recompute when the CSV is missing or older than the study's config, the worker or the model code."""
    csv = sweep_csv(study)
    newest_input = max(p.stat().st_mtime for p in [ROOT / study / 'study_config.py', *CODE])
    return REFRESH or not csv.exists() or csv.stat().st_mtime < newest_input

jobs = {}
for study in STUDIES:                                    # one process per study, all at once
    if stale(study):
        log = open(FIGS / f'{study}_sweeps.log', 'w')
        env = dict(os.environ, MHI_STUDY=str(ROOT / study / 'study_config.py'))
        jobs[study] = (subprocess.Popen([sys.executable, str(WORKER), str(sweep_csv(study))],
                                        cwd=ROOT, env=env, stdout=log, stderr=subprocess.STDOUT), log)
for study, (proc, log) in jobs.items():
    proc.wait(); log.close()
    if proc.returncode:
        raise RuntimeError(f'{study} failed -- see {log.name}')
    print(f'{study}: computed')

data = pd.concat([pd.read_csv(sweep_csv(s)).assign(folder=s) for s in STUDIES], ignore_index=True)
data['label'] = data.metal.map(lambda m: SHORT_NAMES.get(m, m))
if data.groupby('label').folder.nunique().max() > 1:   # two studies on one metal: say which is which
    data['label'] = data.label + ' (' + data.folder + ')'
LABELS = {s: data.loc[data.folder == s, 'label'].iloc[0] for s in STUDIES}

henry = data['level'] == 'L2a'                           # Henry: J·L/ΔP; every other level: J·L/Δ√P
drive = np.where(henry, data.P - data.P_down, np.sqrt(data.P) - np.sqrt(data.P_down))
data['phi'] = data.flux * data.L_total / drive
data['phi_units'] = np.where(henry, UNITS_HENRY, UNITS_SIEVERTS)

# What each study was actually run with, read back from its own results
RUN_WITH = (data.groupby('folder', sort=False)
                [['label', 'metal', 'oxide', 'T_op', 'P_up', 'P_down', 'L_ox0', 'L_m0',
                  'oxide_fp', 'micro_fp', 'defects_fp', 'conditions_fp']].first())
RUN_WITH['ratio'] = RUN_WITH.L_m0 / RUN_WITH.L_ox0
display(RUN_WITH)

def shared(col, text, otherwise):
    """Title text for one setting: its value if every study uses the same one, else the fallback."""
    values = RUN_WITH[col].unique()
    return text(values[0]) if len(values) == 1 else otherwise

SAME_BLOCKS = all(RUN_WITH[c].nunique() == 1 for c in ('oxide_fp', 'micro_fp', 'defects_fp'))
WALL_NOTE = ('same oxide, microstructure and defects' if SAME_BLOCKS
             else 'oxide / microstructure / defects differ by study')

In [ ]:
mpl.rcParams.update({
    'font.size': 14, 'font.weight': 'bold', 'mathtext.default': 'bf',
    'axes.labelweight': 'bold', 'axes.titleweight': 'bold', 'axes.linewidth': 1.8,
    'xtick.direction': 'in', 'ytick.direction': 'in', 'ytick.right': True,
    'xtick.major.width': 1.6, 'ytick.major.width': 1.6,
    'legend.frameon': False, 'legend.fontsize': 11, 'savefig.dpi': 600, 'savefig.bbox': 'tight',
})
INK, GREY, SHADE = '#0b0b0b', '#52514e', '#e1e0d9'

# Regime labels name the dominant path and the limiting step inside it (calculations/wall_labels.py):
# colour = limiting step, hollow = a defect path carries most of the flux, diamond = trap-controlled.
from calculations.wall_labels import describe as look, fitted

UNITS_TEX = {UNITS_SIEVERTS: r'mol m$^{-1}$ s$^{-1}$ Pa$^{-1/2}$', UNITS_HENRY: r'mol m$^{-1}$ s$^{-1}$ Pa$^{-1}$'}

LEVEL_TITLES = {
    'L1':    'Level 1: Perfect metal (lattice diffusion)',
    'L2a':   'Level 2a: Perfect oxide alone (molecular H$_2$)',
    'L2b':   'Level 2b: Perfect oxide + perfect metal',
    'L3':    'Level 3: Defective oxide + perfect metal',
    'L4':    'Level 4: Defective metal (traps + grain boundaries)',
    'L5':    'Level 5: Defective oxide + defective metal',
    'L1L6':  'Level 1+6: Perfect metal + surface kinetics',
    'L2aL6': 'Level 2a+6: Perfect oxide alone + surface kinetics',
    'L2bL6': 'Level 2b+6: Perfect oxide + perfect metal + surface kinetics',
    'L3L6':  'Level 3+6: Defective oxide + perfect metal + surface kinetics',
    'L4L6':  'Level 4+6: Perfect oxide + defective metal + surface kinetics',
    'L5L6':  'Level 5+6: Defective oxide + defective metal + surface kinetics',
}

In [ ]:
LINES = ['-', '--', ':']                                 # line style per metal, in STUDIES order
EXPONENT = {'T': 'E_a (kJ/mol)', 'P': 'n (J) / slope (Φ_app)', 'L': 'm (J) / slope (Φ_app)',
            'f': 'n (J, Φ_app ∝ f^n)'}

def framed(nrows, figsize, sharex=True):
    """Figure with two legend strips under the title and an nrows x 2 grid of panels below them."""
    fig  = plt.figure(figsize=figsize, layout='constrained')
    gs   = fig.add_gridspec(nrows + 2, 2, height_ratios=[0.001, 0.001] + [1] * nrows)
    keys = [fig.add_subplot(gs[k, :]) for k in (0, 1)]
    for k in keys:
        k.axis('off')
    ax = np.empty((nrows, 2), dtype=object)
    for r in range(nrows):
        for c in range(2):
            ax[r, c] = fig.add_subplot(gs[r + 2, c], sharex=ax[0, c] if (r and sharex) else None)
    return fig, keys, ax

def shared_legend(keys, axes, lines):
    """One legend per figure: the markers (by regime) on the first strip, the lines on the second."""
    marks = {}
    for a in axes:
        for h, label in zip(*a.get_legend_handles_labels()):
            marks.setdefault(label, h)
    keys[0].legend(list(marks.values()), list(marks), title='Simulation', loc='center', ncol=min(len(marks), 2))
    keys[1].legend(handles=lines, loc='center', ncol=len(lines))

def metal_lines(labels):
    return [Line2D([], [], color=INK, lw=2.4, ls=LINES[ci], label=label) for ci, label in enumerate(labels)]

def right_end(xv, y, tag):
    """(x, y, tag) at a curve's right-hand end, for label_ends()."""
    ok = np.flatnonzero(np.isfinite(y))
    i = ok[np.argmax(xv[ok])]
    return xv[i], y[i], tag

def label_ends(ax, ends, gap=0.05):
    """Name each curve at its right-hand end, pushed apart where curves coincide. Call after the limits are set."""
    to_axes = ax.transAxes.inverted()
    stack = sorted((to_axes.transform(ax.transData.transform((x, y)))[1], x, tag) for x, y, tag in ends)
    for k in range(1, len(stack)):                       # push each label clear of the one below it
        stack[k] = (max(stack[k][0], stack[k - 1][0] + gap),) + stack[k][1:]
    shift = max(0.0, stack[-1][0] - 0.97) if stack else 0.0   # ...and keep the top one inside the panel
    for yf, x, tag in stack:
        ax.annotate(tag, (x, yf - shift), xycoords=('data', 'axes fraction'), xytext=(10, 0),
                    textcoords='offset points', va='center', fontsize=11, color=INK)

def fit_table(rows):
    """Every fitted line as one row: shown under the figure, and returned for the CSV."""
    table = pd.DataFrame(rows).replace({r'\$_2\$': '₂'}, regex=True)
    with pd.option_context('display.float_format', '{:.4g}'.format, 'display.width', 220,
                           'display.max_columns', 20):
        display(table)
    return table

def fit(kind, xs, y):
    """kind 'T': Arrhenius on xs = 1000/T, E_a in kJ/mol. Otherwise a power law y = A·xs^k (xs in SI)."""
    if kind == 'T':
        f = stats.linregress(xs, np.log(y))
        return -f.slope * R, np.exp(f.intercept), f.rvalue**2, np.exp(f.intercept + f.slope * xs)
    f = stats.linregress(np.log(xs), np.log(y))
    return f.slope, np.exp(f.intercept), f.rvalue**2, np.exp(f.intercept) * xs**f.slope

def panel(ax, kind, xs, x_axis, window, y, regime, by_regime, rows, info, ls):
    """One metal's points and fits, one group or one per regime; every fit becomes a table row."""
    plot = ax.semilogy if kind == 'T' else ax.loglog
    for reg in (dict.fromkeys(regime) if by_regime else ['all']):
        m = (regime == reg) if by_regime else np.full(len(y), True)
        name, c, mk, *fill = look(reg) if by_regime else ('All points (single fit)', GREY, 'o')
        hollow = fill == ['hollow']
        plot(x_axis[m], y[m], mk, ms=8, color=c, mfc='white' if hollow else c,
             mec=c if hollow else 'white', mew=2 if hollow else 1, label=name)
        if not fitted(reg) or m.sum() < 3:
            continue                                     # drawn, not fitted
        k, A, r2, line = fit(kind, xs[m], y[m])
        plot(x_axis[m], line, ls, lw=2.6, color=c if by_regime else INK)
        rows.append({**info, 'fit': name if by_regime else 'single fit',
                     'window': f'{window[m].min():.4g}–{window[m].max():.4g}', 'points': int(m.sum()),
                     EXPONENT[kind]: k, 'A (SI)': A, 'R²': r2})
    ax.grid(color=SHADE)

def curves(level, sweep):
    """[(metal label, that metal's rows of this sweep, in order along it), ...]; empty if the layer is absent."""
    d = data[(data.level == level) & (data.sweep == sweep)]
    key = {'temperature': 'T', 'pressure': 'P', 'oxide': 'L_ox', 'metal': 'L_m', 'total': 'L_total', 'pinholes': 'f'}[sweep]
    return [(LABELS[s], g.sort_values(key)) for s in STUDIES if len(g := d[d.folder == s])]

def no_layer(ax, layer):
    ax.text(0.5, 0.5, f'no {layer} in this level', transform=ax.transAxes,
            ha='center', va='center', fontsize=15, color=GREY)
    ax.set_xticks([]); ax.set_yticks([])

In [ ]:
def sweep_figure(level, sweep):
    """2x2 figure for the temperature or pressure sweep, one curve per metal, and its fit table."""
    kind, cs = ('T' if sweep == 'temperature' else 'P'), curves(level, sweep)
    units = cs[0][1].phi_units.iloc[0]
    fig, keys, ax = framed(2, (14, 13))
    rows = []
    for r, (col_y, qty, ylab) in enumerate((('flux', 'J',       r'Flux $J$ (mol m$^{-2}$ s$^{-1}$)'),
                                            ('phi',  'Phi_app', rf'$\Phi_{{app}}$ ({UNITS_TEX[units]})'))):
        ys = np.concatenate([d[col_y].to_numpy() for _, d in cs])
        for col in (0, 1):
            p, ends, xall = ax[r, col], [], []
            for ci, (label, d) in enumerate(cs):
                window = d['T' if kind == 'T' else 'P'].to_numpy()
                xs = 1000 / window if kind == 'T' else window
                y = d[col_y].to_numpy()
                if kind == 'P' and r == 0 and ci == 1:     # reference slopes n = 1 and 0.5, through the middle metal
                    i = len(xs) // 2
                    for n in (1.0, 0.5):
                        p.loglog(xs, y[i] * (xs / xs[i])**n, '-', color=GREY, lw=1.0, alpha=0.6, zorder=0)
                panel(p, kind, xs, xs, window, y, d.regime.to_numpy(), col == 1, rows,
                      {'quantity': qty, 'metal': label}, LINES[ci])
                ends.append(right_end(xs, y, label))
                xall.append(xs)
            xall = np.concatenate(xall)                                       # every study's own range
            if kind == 'T':
                p.set_xlim(xall.min() - 0.03, xall.max() + 0.32)             # room for the end labels
            else:
                p.set_xlim(xall.min() / 3, xall.max() * 1000)
                p.set_ylim(np.nanmin(ys) / 3, np.nanmax(ys) * 3)            # keep the guides from stretching it
            label_ends(p, ends)
            p.set_ylabel(ylab)
    for p in ax[0]:
        p.tick_params(labelbottom=False)
        if kind == 'T':
            top = p.secondary_xaxis('top', functions=(lambda v: 1000/v - 273.15, lambda c: 1000/(c + 273.15)))
            top.set_xticks([300, 400, 500, 600, 700, 800, 1000])           # 900 °C would collide with 1000 °C
            top.set_xlabel('Temperature (°C)')
    for p in ax[1]:
        p.set_xlabel(r'1000/$T$ (K$^{-1}$)' if kind == 'T' else r'Upstream pressure $P_{up}$ (Pa)')
    fit_name = 'Arrhenius' if kind == 'T' else 'power-law'
    ax[0, 0].set_title(f'Single {fit_name} fit (all {sweep}s)')
    ax[0, 1].set_title(f'Regime-resolved {fit_name} fits')
    lines = metal_lines([label for label, _ in cs])
    if kind == 'P':
        lines.append(Line2D([], [], color=GREY, lw=1.0, alpha=0.6, label='Reference slopes $n$ = 1 and 0.5'))
    shared_legend(keys, ax.flat, lines)
    drive = r'\Delta P' if units == UNITS_HENRY else r'\Delta\sqrt{P}'
    fixed = (shared('P_up', lambda v: rf'$P_{{up}}$ = {v:.0e} Pa', 'each study at its own $P_{up}$') if kind == 'T'
             else shared('T_op', lambda v: rf'$T$ = {v:.0f} K', 'each study at its own $T_{op}$'))
    fig.suptitle(f'{LEVEL_TITLES[level]}\n{len(cs)} studies, {WALL_NOTE}\n{fixed},   '
                 + rf'$\Phi_{{app}} = J\,L/{drive}$', fontsize=18, fontweight='bold')
    plt.show()
    return fig, fit_table(rows)

THICKNESS = {   # sweep: (x-axis label, column swept, metres -> axis units)
    'oxide': (r'Oxide thickness $L_{ox}$ (µm)',     'L_ox',    1e6),
    'metal': (r'Metal thickness $L_m$ (mm)',        'L_m',     1e3),
    'total': (r'Total thickness $L_{total}$ (mm)',  'L_total', 1e3),
}
THICKNESS_TITLES = {   # the fixed layer is each study's own; stated once only if every study agrees
    'oxide': 'Oxide thickness swept (' + shared('L_m0', lambda v: f'metal fixed at {v*1e3:g} mm',
                                                "metal at each study's own L_metal") + ')',
    'metal': 'Metal thickness swept (' + shared('L_ox0', lambda v: f'oxide fixed at {v*1e6:g} µm',
                                                "oxide at each study's own L_oxide") + ')',
    'total': 'Whole wall scaled (' + shared('ratio', lambda v: f'oxide:metal fixed at 1:{v:.0f}',
                                            "each study's own oxide:metal ratio") + ')',
}

def thickness_figure(level):
    """3x2 figure, one row per thickness sweep and one curve per metal, and its fit table."""
    fig, keys, ax = framed(3, (14, 19), sharex=False)
    rows, units, labels = [], UNITS_SIEVERTS, [LABELS[s] for s in STUDIES]
    for i, (sweep, (x_label, x_col, to_axis)) in enumerate(THICKNESS.items()):
        cs = curves(level, sweep)
        for j, (col_y, qty) in enumerate((('flux', 'J'), ('phi', 'Phi_app'))):
            p = ax[i, j]
            p.set_title(THICKNESS_TITLES[sweep], fontsize=13)
            if not cs:
                no_layer(p, sweep)
                continue
            units, ends, xall = cs[0][1].phi_units.iloc[0], [], []
            for ci, (label, d) in enumerate(cs):
                xs, y = d[x_col].to_numpy(), d[col_y].to_numpy()
                panel(p, 'L', xs, xs * to_axis, xs, y, d.regime.to_numpy(), True, rows,
                      {'quantity': qty, 'sweep': sweep, 'metal': label}, LINES[ci])
                ends.append(right_end(xs * to_axis, y, label))
                xall.append(xs * to_axis)
            xall = np.concatenate(xall)                                       # every study's own range
            p.set_xlim(xall.min() / 1.5, xall.max() * 15)                   # room for the end labels
            label_ends(p, ends)
            p.set_xlabel(x_label)
            p.set_ylabel(r'Flux $J$ (mol m$^{-2}$ s$^{-1}$)' if qty == 'J'
                         else rf'$\Phi_{{app}}$ ({UNITS_TEX[units]})')
    shared_legend(keys, ax.flat, metal_lines(labels))
    drive = r'\Delta P' if units == UNITS_HENRY else r'\Delta\sqrt{P}'
    fixed = (shared('T_op', lambda v: rf'$T$ = {v:.0f} K', 'each study at its own $T_{op}$') + ',   '
             + shared('P_up', lambda v: rf'$P_{{up}}$ = {v:.0e} Pa', 'own $P_{up}$'))
    fig.suptitle(f'{LEVEL_TITLES[level]}\n{len(labels)} studies, {WALL_NOTE}\n{fixed},   '
                 + rf'$J \propto L^m$,   $\Phi_{{app}} = J\,L_{{total}}/{drive}$',
                 fontsize=18, fontweight='bold')
    plt.show()
    return fig, fit_table(rows)

def pinhole_figure(level):
    """1x2 figure (J, Φ_app against pinhole area fraction, pinholes only), one curve per metal, and its fit table."""
    cs = curves(level, 'pinholes')
    fig, keys, ax = framed(1, (14, 8.5))
    rows = []
    for j, (col_y, qty, ylab) in enumerate((('flux', 'J',       r'Flux $J$ (mol m$^{-2}$ s$^{-1}$)'),
                                            ('phi',  'Phi_app', rf'$\Phi_{{app}}$ ({UNITS_TEX[UNITS_SIEVERTS]})'))):
        p, ends, xall = ax[0, j], [], []
        for ci, (label, d) in enumerate(cs):
            xs, y = d['f'].to_numpy(), d[col_y].to_numpy()
            panel(p, 'f', xs, xs, xs, y, d.regime.to_numpy(), True, rows,
                  {'quantity': qty, 'metal': label}, LINES[ci])
            ends.append(right_end(xs, y, label))
            xall.append(xs)
        xall = np.concatenate(xall)
        p.set_xlim(xall.min() / 1.5, xall.max() * 15)                       # room for the end labels
        label_ends(p, ends)
        p.set_xlabel('Pinhole area fraction $f$')
        p.set_ylabel(ylab)
    ax[0, 0].set_title('Flux')
    ax[0, 1].set_title('Apparent permeability')
    shared_legend(keys, ax.flat, metal_lines([label for label, _ in cs]))
    fixed = (shared('T_op', lambda v: rf'$T$ = {v:.0f} K', 'each study at its own $T_{op}$') + ',   '
             + shared('P_up', lambda v: rf'$P_{{up}}$ = {v:.0e} Pa', 'own $P_{up}$'))
    fig.suptitle(f'{LEVEL_TITLES[level]}, pinholes only\n{len(cs)} studies\n{fixed},   '
                 + rf'$J = (1-f)\,J_i + f\,J_p$,   $\Phi_{{app}} = J\,L_{{total}}/\Delta\sqrt{{P}}$',
                 fontsize=18, fontweight='bold')
    plt.show()
    return fig, fit_table(rows)


In [ ]:
for level in LEVEL_TITLES:
    fig, fits = sweep_figure(level, 'temperature')
    save_figure(fig, f"{level}_temperature_by_study", FIGS)
    fits.to_csv(FIGS / f"{level}_temperature_by_study_fits.csv", index=False)

In [ ]:
for level in LEVEL_TITLES:
    fig, fits = sweep_figure(level, 'pressure')
    save_figure(fig, f"{level}_pressure_by_study", FIGS)
    fits.to_csv(FIGS / f"{level}_pressure_by_study_fits.csv", index=False)

In [ ]:
for level in LEVEL_TITLES:
    fig, fits = thickness_figure(level)
    save_figure(fig, f"{level}_thickness_by_study", FIGS)
    fits.to_csv(FIGS / f"{level}_thickness_by_study_fits.csv", index=False)

In [ ]:
for level in [lv for lv in LEVEL_TITLES if ((data.level == lv) & (data.sweep == 'pinholes')).any()]:
    fig, fits = pinhole_figure(level)
    save_figure(fig, f"{level}_pinholes_by_study", FIGS)
    fits.to_csv(FIGS / f"{level}_pinholes_by_study_fits.csv", index=False)